In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

from sklearn.metrics import accuracy_score, classification_report

In [3]:
df = pd.read_csv("../data/cleaned/spam.csv")

df = df.dropna(subset=['clean_message'])
df = df[df['clean_message'].str.strip() != ""]

In [4]:
encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["label"])

In [5]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(df["clean_message"])

X = tokenizer.texts_to_sequences(df["clean_message"])

In [6]:
# Ensure tokenizer exists and is fitted, then create padded sequences
try:
	tokenizer
except NameError:
	tokenizer = Tokenizer(num_words=5000)

if not hasattr(tokenizer, "word_index") or len(tokenizer.word_index) == 0:
	tokenizer.fit_on_texts(df["clean_message"])

sequences = tokenizer.texts_to_sequences(df["clean_message"])
X = pad_sequences(sequences, maxlen=100)

In [7]:
X = pad_sequences(X, maxlen=100)

y = df["label"]

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df["clean_message"])

In [9]:
import pickle

with open("../models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved successfully!")

Tokenizer saved successfully!


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
gru_model = Sequential()

gru_model.add(Embedding(input_dim=5000, output_dim=64))

gru_model.add(GRU(64))

gru_model.add(Dropout(0.5))

gru_model.add(Dense(1, activation="sigmoid"))

gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
history = gru_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.9093 - loss: 0.2592 - val_accuracy: 0.9686 - val_loss: 0.1028
Epoch 2/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.9857 - loss: 0.0511 - val_accuracy: 0.9764 - val_loss: 0.0893
Epoch 3/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.9944 - loss: 0.0209 - val_accuracy: 0.9787 - val_loss: 0.0734
Epoch 4/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9975 - loss: 0.0108 - val_accuracy: 0.9809 - val_loss: 0.1106
Epoch 5/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.9986 - loss: 0.0061 - val_accuracy: 0.9562 - val_loss: 0.1348


In [13]:
loss, accuracy = gru_model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9587 - loss: 0.1125
Test Accuracy: 0.958707332611084


In [14]:
y_pred = (gru_model.predict(X_test) > 0.5).astype(int)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


In [15]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.96      0.98       979
           1       0.76      0.96      0.85       135

    accuracy                           0.96      1114
   macro avg       0.88      0.96      0.91      1114
weighted avg       0.97      0.96      0.96      1114



In [16]:
gru_model.save("../models/gru_model.keras")

test_messages = [
    "FREE entry into our weekly competition. Text WIN to 80085 now.",
    "Hi, are we still meeting at 5 PM today?"
]

seq = tokenizer.texts_to_sequences(test_messages)
pad = pad_sequences(seq, maxlen=100)

predictions = gru_model.predict(pad)

for msg, pred in zip(test_messages, predictions):
    print(msg)
    print("Prediction:", pred[0])
    print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
FREE entry into our weekly competition. Text WIN to 80085 now.
Prediction: 0.9991969

Hi, are we still meeting at 5 PM today?
Prediction: 0.0004319027



In [18]:
test_messages = [
    "FREE entry into our weekly competition. Text WIN to 80085 now.",
    "Hi, are we still meeting at 5 PM today?"
]

test_sequences = tokenizer.texts_to_sequences(test_messages)
test_padded = pad_sequences(test_sequences, maxlen=100)

predictions = gru_model.predict(test_padded)

for msg, pred in zip(test_messages, predictions):
    print(msg)
    print("Probability:", pred)
    print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
FREE entry into our weekly competition. Text WIN to 80085 now.
Probability: [0.9991969]

Hi, are we still meeting at 5 PM today?
Probability: [0.0004319]



In [19]:
predictions = gru_model.predict(test_padded)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


In [20]:

print(predictions)
print(predictions.shape)

[[9.991969e-01]
 [4.319027e-04]]
(2, 1)


In [22]:

test_sequences = tokenizer.texts_to_sequences(test_messages)


In [23]:
print(test_sequences)

[[8, 363, 404, 1436, 20, 100], [49, 35, 214, 273, 27]]
